[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week08.ipynb)

# Week 8: Convolutional Networks I
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Convolution, pooling, and feature maps; building a CNN image classifier.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Build and train a CNN image classifier.
- Understand convolution, pooling, and feature maps.
- Compute how shapes and parameters change layer by layer.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. What a convolution does
A small learned filter slides over the input, sharing weights across positions.

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1)
x = torch.randn(1, 1, 8, 8)
print("input ", tuple(x.shape), "-> output", tuple(conv(x).shape))   # 4 feature maps, same H,W (padding=1)
print("filter weights:", tuple(conv.weight.shape), "(out, in, kh, kw)")

## 2. Output size and parameter count
out = floor((in + 2p - k)/s) + 1 ; conv params = (k*k*Cin + 1)*Cout.

In [ ]:
c = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
print("output:", tuple(c(torch.randn(1, 3, 32, 32)).shape), " formula (32+2-3)/1+1 = 32")
print("params:", sum(p.numel() for p in c.parameters()), "= (3*3*3 + 1) * 16 =", (3 * 3 * 3 + 1) * 16)

## 3. Build a CNN and trace shapes
Stack conv/pool blocks; channels grow while spatial size shrinks.

In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 28 -> 14
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 14 -> 7
    nn.Flatten(), nn.Linear(16 * 7 * 7, 10))
x = torch.randn(4, 1, 28, 28)
for layer in cnn:
    x = layer(x); print(f"{layer.__class__.__name__:9s}", tuple(x.shape))

## 4. Train on FashionMNIST
A couple of hundred steps is enough to see it learn (downloads ~30 MB).

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
train = datasets.FashionMNIST("./data", train=True, download=True, transform=transforms.ToTensor())
dl = DataLoader(train, batch_size=128, shuffle=True)
model = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Flatten(), nn.Linear(8 * 14 * 14, 10)).to(device)
opt = torch.optim.Adam(model.parameters(), 1e-3); f = nn.CrossEntropyLoss()
for step, (xb, yb) in enumerate(dl):
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(); loss = f(model(xb), yb); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step}: loss {loss.item():.3f}")
    if step == 200:
        break

## 5. Visualize the first-layer feature maps
Each map is one filter's response across the image.

In [ ]:
fmap = model[0](xb[:1]).detach().cpu()
fig, ax = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    ax[i].imshow(fmap[0, i]); ax[i].axis("off")
plt.suptitle("First-conv feature maps"); plt.show()

---
Student materials for this week: the lab handout (`labs/week08.html`) and the curated references (`references/week08.html`) in the course site.